# 03 Búsqueda vectorial con TF-IDF
- **Estudiante**: J. L. Andino
- **Fecha de Entrega:** Miercoles 11 de noviembre de 2026
- **Curso:** GR1CC
- **Carrera:** Computación
- **Docente:** Prof. Iván Carrera

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
# ============ Celda 1: Corpus y estadísticas (Sección 1) ============
import math
import re
import sys
import unicodedata
from abc import ABC, abstractmethod
from collections import Counter
from dataclasses import dataclass

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("gutenberg", quiet=True)


class TextNormalizer:
    """Preprocesamiento de la práctica 02: minúsculas -> NFKD sin diacríticos -> tokens [a-z0-9]+."""
    _TOKEN_RE = re.compile(r"[a-z0-9]+")

    def normalize(self, text: str) -> str:
        text = unicodedata.normalize("NFKD", text.lower())
        return text.encode("ascii", "ignore").decode("ascii")

    def tokenize(self, text: str) -> list[str]:
        return [sys.intern(t) for t in self._TOKEN_RE.findall(self.normalize(text))]


@dataclass(frozen=True)
class Document:
    doc_id: str
    title: str
    raw: str             # texto original: solo para mostrar fragmentos
    tokens: list[str]    # tokens procesados: única base de todas las técnicas


TITLES = {
    "austen-emma.txt": "Emma",
    "austen-persuasion.txt": "Persuasion",
    "austen-sense.txt": "Sense and Sensibility",
    "bible-kjv.txt": "The King James Bible",
    "blake-poems.txt": "Poems",
    "bryant-stories.txt": "Stories to Tell to Children",
    "burgess-busterbrown.txt": "The Adventures of Buster Bear",
    "carroll-alice.txt": "Alice's Adventures in Wonderland",
    "chesterton-ball.txt": "The Ball and the Cross",
    "chesterton-brown.txt": "The Wisdom of Father Brown",
    "chesterton-thursday.txt": "The Man Who Was Thursday",
    "edgeworth-parents.txt": "The Parent's Assistant",
    "melville-moby_dick.txt": "Moby Dick",
    "milton-paradise.txt": "Paradise Lost",
    "shakespeare-caesar.txt": "Julius Caesar",
    "shakespeare-hamlet.txt": "Hamlet",
    "shakespeare-macbeth.txt": "Macbeth",
    "whitman-leaves.txt": "Leaves of Grass",
}


def load_documents(normalizer: TextNormalizer) -> list[Document]:
    docs = []
    for file_id in gutenberg.fileids():
        raw = gutenberg.raw(file_id)
        docs.append(Document(file_id, TITLES[file_id], raw, normalizer.tokenize(raw)))
    return docs


normalizer = TextNormalizer()
docs = load_documents(normalizer)

lengths = np.array([len(d.tokens) for d in docs])
vocab_size = len({t for d in docs for t in d.tokens})

display(pd.DataFrame({
    "Identificador": [d.doc_id for d in docs],
    "Título": [d.title for d in docs],
    "Términos únicos": [len(set(d.tokens)) for d in docs],
}))
print(f"Documentos: {len(docs)} | Vocabulario: {vocab_size:,} términos")
print(f"Longitud (tokens) -> mínima: {lengths.min():,} | media: {lengths.mean():,.1f} | máxima: {lengths.max():,}")

,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,7103
1,austen-persuasion.txt,Persuasion,5777
2,austen-sense.txt,Sense and Sensibility,6336
3,bible-kjv.txt,The King James Bible,12744
4,blake-poems.txt,Poems,1512
5,bryant-stories.txt,Stories to Tell to Children,3836
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1527
7,carroll-alice.txt,Alice's Adventures in Wonderland,2570
8,chesterton-ball.txt,The Ball and the Cross,8234
9,chesterton-brown.txt,The Wisdom of Father Brown,7739


Documentos: 18 | Vocabulario: 41,838 términos
Longitud (tokens) -> mínima: 6,936 | media: 122,178.2 | máxima: 854,046


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
# ============ Celda 2: Funciones TF, DF, IDF y TF-IDF (Sección 2) ============
def compute_tf(tokens: list[str]) -> Counter:
    """TF(t,d) = f(t,d): número de apariciones del término en el documento."""
    return Counter(tokens)


def compute_df(token_lists: list[list[str]]) -> Counter:
    """DF(t): documentos del corpus que contienen t (cada documento cuenta una sola vez)."""
    df = Counter()
    for tokens in token_lists:
        df.update(set(tokens))
    return df


def compute_idf(df: Counter, n_docs: int) -> dict[str, float]:
    """IDF(t) = ln(N / DF(t)) + 1  (fórmula de scikit-learn con smooth_idf=False)."""
    return {term: math.log(n_docs / count) + 1 for term, count in df.items()}


def compute_tfidf(tf: Counter, idf: dict[str, float]) -> dict[str, float]:
    """w(t,d) = TF(t,d) * IDF(t), solo para los términos presentes en el documento."""
    return {term: freq * idf[term] for term, freq in tf.items()}


# Ejemplo mínimo comprobable a mano: 2 documentos x 3 términos = 6 pesos TF-IDF
toy_docs = [["gato", "perro", "gato"], ["perro", "pez"]]
toy_tfs = [compute_tf(d) for d in toy_docs]
toy_idf = compute_idf(compute_df(toy_docs), len(toy_docs))
toy_terms = sorted(toy_idf)

pd.DataFrame(
    [[compute_tfidf(tf, toy_idf).get(t, 0.0) for t in toy_terms] for tf in toy_tfs],
    index=["d1: gato perro gato", "d2: perro pez"], columns=toy_terms,
)

,gato,perro,pez
d1: gato perro gato,3.386294,1.0,0.000000
d2: perro pez,0.000000,1.0,1.693147


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
# ============ Celda 3: TF-IDF del corpus (Sección 3) ============
token_lists = [d.tokens for d in docs]

# 1. Vocabulario con orden definido (alfabético)
vocabulary = sorted({t for tokens in token_lists for t in tokens})
term_index = {t: i for i, t in enumerate(vocabulary)}

# 2-4. TF, DF (una vez por documento) e IDF
tfs = [compute_tf(tokens) for tokens in token_lists]
df = compute_df(token_lists)
idf = compute_idf(df, len(docs))


# 5. Matriz documento-término
def build_tfidf_matrix(tfs: list[Counter], idf: dict[str, float], term_index: dict[str, int]) -> np.ndarray:
    matrix = np.zeros((len(tfs), len(term_index)))
    for row, tf in enumerate(tfs):
        for term, weight in compute_tfidf(tf, idf).items():
            matrix[row, term_index[term]] = weight
    return matrix


X_manual = build_tfidf_matrix(tfs, idf, term_index)
print(f"Matriz TF-IDF: {X_manual.shape[0]} documentos x {X_manual.shape[1]:,} términos")

Matriz TF-IDF: 18 documentos x 41,838 términos


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
# ============ Celda 4: Verificación con TfidfVectorizer (Sección 4) ============
def identity_analyzer(tokens: list[str]) -> list[str]:
    """Entrega al vectorizador los mismos tokens ya procesados (sin re-tokenizar)."""
    return tokens


vectorizer = TfidfVectorizer(
    analyzer=identity_analyzer,
    vocabulary=term_index,        # mismo vocabulario y mismo orden de columnas
    use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None,
)
X_sklearn = vectorizer.fit_transform(token_lists).toarray()

max_abs_diff = np.abs(X_manual - X_sklearn).max()
print(f"Máxima diferencia absoluta: {max_abs_diff:.3e}")
assert max_abs_diff < 1e-9, "Revisar preprocesamiento, orden de columnas, fórmula de IDF o normalización"

Máxima diferencia absoluta: 0.000e+00


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [5]:
# ============ Celda 5: Consultas y similitud coseno (Sección 5) ============
TOLERANCE = 1e-9


@dataclass(frozen=True)
class QuerySpec:
    query_id: str
    need: str     # información que espera encontrar el usuario
    text: str


QUERIES = [
    QuerySpec("q1", "Pasajes sobre la caza de la ballena blanca y el capitán obsesionado", "captain ahab white whale sea ship"),
    QuerySpec("q2", "Novelas de costumbres: matrimonio, amor y vida social", "marriage love family society gentleman lady"),
    QuerySpec("q3", "Tragedias sobre asesinato, ambición y realeza", "murder ambition king queen crown tragedy"),
    QuerySpec("q4", "Textos religiosos o épicos sobre la creación y la caída del hombre", "god creation heaven earth fall adam eve"),
]


def cosine_manual(q: np.ndarray, d: np.ndarray) -> float:
    """sim(q,d) = (q·d) / (||q|| ||d||) con producto punto y normas explícitas."""
    norm_q, norm_d = np.sqrt(np.dot(q, q)), np.sqrt(np.dot(d, d))
    return float(np.dot(q, d) / (norm_q * norm_d)) if norm_q and norm_d else 0.0


class Scorer(ABC):
    """Contrato común: cualquier técnica de ranking devuelve una puntuación por documento."""

    @abstractmethod
    def query_vector(self, text: str) -> np.ndarray: ...

    @abstractmethod
    def scores(self, text: str) -> np.ndarray: ...


class ManualCosineScorer(Scorer):
    def __init__(self, matrix, term_index, idf, normalizer):
        self._matrix, self._term_index, self._idf, self._normalizer = matrix, term_index, idf, normalizer

    def query_vector(self, text: str) -> np.ndarray:
        """Vector de la consulta con el vocabulario e IDF del corpus (no se recalcula IDF)."""
        vector = np.zeros(len(self._term_index))
        for term, freq in compute_tf(self._normalizer.tokenize(text)).items():
            if term in self._term_index:                  # términos fuera del vocabulario: peso 0
                vector[self._term_index[term]] = freq * self._idf[term]
        return vector

    def scores(self, text: str) -> np.ndarray:
        q = self.query_vector(text)
        return np.array([cosine_manual(q, row) for row in self._matrix])


class SklearnCosineScorer(Scorer):
    def __init__(self, vectorizer, matrix, normalizer):
        self._vectorizer, self._matrix, self._normalizer = vectorizer, matrix, normalizer

    def query_vector(self, text: str) -> np.ndarray:
        return self._vectorizer.transform([self._normalizer.tokenize(text)]).toarray()[0]

    def scores(self, text: str) -> np.ndarray:
        return cosine_similarity(self.query_vector(text).reshape(1, -1), self._matrix)[0]


manual_scorer = ManualCosineScorer(X_manual, term_index, idf, normalizer)
sklearn_scorer = SklearnCosineScorer(vectorizer, X_sklearn, normalizer)

# Comparación explícita vs scikit-learn
rows = []
for spec in QUERIES:
    rows.append({
        "consulta": spec.query_id,
        "términos en vocabulario": int((manual_scorer.query_vector(spec.text) > 0).sum()),
        "max |Δ vector consulta|": np.abs(manual_scorer.query_vector(spec.text) - sklearn_scorer.query_vector(spec.text)).max(),
        "max |Δ similitud|": np.abs(manual_scorer.scores(spec.text) - sklearn_scorer.scores(spec.text)).max(),
    })
display(pd.DataFrame(rows))


def explain_pair(q: np.ndarray, d: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Términos compartidos y productos parciales + resumen (producto punto, normas, similitud)."""
    shared = np.flatnonzero((q > 0) & (d > 0))
    terms = pd.DataFrame({
        "Término compartido": [vocabulary[i] for i in shared],
        "q_i": q[shared], "d_i": d[shared], "Producto parcial q_i·d_i": q[shared] * d[shared],
    }).sort_values("Producto parcial q_i·d_i", ascending=False)
    summary = pd.DataFrame([{
        "Producto punto": np.dot(q, d), "||q||": np.linalg.norm(q),
        "||d||": np.linalg.norm(d), "Similitud": cosine_manual(q, d),
    }])
    return terms, summary


# Desglose para una combinación consulta-documento: q1 contra su documento mejor puntuado
spec = QUERIES[0]
best_doc = int(np.argmax(manual_scorer.scores(spec.text)))
terms_df, summary_df = explain_pair(manual_scorer.query_vector(spec.text), X_manual[best_doc])
print(f"Consulta {spec.query_id} vs documento '{docs[best_doc].title}'")
display(terms_df)
display(summary_df)

,consulta,términos en vocabulario,max |Δ vector consulta|,max |Δ similitud|
0,q1,6,0.0,5.273559e-16
1,q2,6,0.0,2.602085e-17
2,q3,6,0.0,1.110223e-16
3,q4,7,0.0,1.804112e-16


Consulta q1 vs documento 'Moby Dick'


,Término compartido,q_i,d_i,Producto parcial q_i·d_i
4,whale,2.098612,2572.898666,5399.516758
0,ahab,3.197225,1633.781759,5223.567194
1,captain,1.693147,557.045422,943.159886
3,ship,1.325422,686.568803,909.993671
2,sea,1.117783,508.591281,568.494706
5,white,1.117783,314.097033,351.092335


,Producto punto,||q||,||d||,Similitud
0,13395.824551,4.66356,21539.90611,0.133355


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
# ============ Celda 6: Ranking top-5 con TF-IDF (Sección 6) ============
def make_snippet(doc: Document, q: np.ndarray, d: np.ndarray, width: int = 160) -> str:
    """Fragmento del texto original alrededor del término que más aporta a la similitud."""
    contributions = q * d
    if contributions.max() > 0:
        match = re.search(rf"\b{re.escape(vocabulary[int(np.argmax(contributions))])}\b", doc.raw, re.IGNORECASE)
        if match:
            start = max(0, match.start() - width // 2)
            return "..." + " ".join(doc.raw[start:start + width].split()) + "..."
    return " ".join(doc.raw[:width].split()) + "..."


def rank_top_k(spec: QuerySpec, scorer: Scorer, k: int = 5) -> pd.DataFrame:
    scores = scorer.scores(spec.text)
    q = scorer.query_vector(spec.text)
    top = np.argsort(-scores, kind="stable")[:k]
    return pd.DataFrame({
        "Posición": range(1, len(top) + 1),
        "Identificador": [docs[i].doc_id for i in top],
        "Título": [docs[i].title for i in top],
        "Puntuación": [round(float(scores[i]), 4) for i in top],
        "Fragmento": [make_snippet(docs[i], q, X_manual[i]) for i in top],
    })


pd.set_option("display.max_colwidth", 200)
for spec in QUERIES:
    explicit, library = rank_top_k(spec, manual_scorer), rank_top_k(spec, sklearn_scorer)
    assert np.allclose(manual_scorer.scores(spec.text), sklearn_scorer.scores(spec.text), atol=TOLERANCE)
    assert explicit["Identificador"].tolist() == library["Identificador"].tolist()
    print(f"\n{spec.query_id}: {spec.need}\n    consulta: '{spec.text}'")
    display(explicit)


q1: Pasajes sobre la caza de la ballena blanca y el capitán obsesionado
    consulta: 'captain ahab white whale sea ship'


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.1334,"...""While you take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the le..."
1,2,austen-persuasion.txt,Persuasion,0.0242,"...th, the former curate of Monkford, however suspicious appearances may be, but a Captain Frederick Wentworth, his brother, who being made commander in consequenc..."
2,3,whitman-leaves.txt,Leaves of Grass,0.0087,"...chant of battles, I above all promote brave soldiers. } In Cabin'd Ships at Sea In cabin'd ships at sea, The boundless blue on every side expanding, With w..."
3,4,bryant-stories.txt,Stories to Tell to Children,0.0069,"...t, unless somebody picked him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on t..."
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.0068,"...eminent criminologist and specialist in certain moral disorders, lay along the sea-front at Scarborough, in a series of very large and well-lighted french wind..."



q2: Novelas de costumbres: matrimonio, amor y vida social
    consulta: 'marriage love family society gentleman lady'


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-persuasion.txt,Persuasion,0.0245,"...r of James Stevenson, Esq. of South Park, in the county of Gloucester, by which lady (who died 1800) he has issue Elizabeth, born June 1, 1785; Anne, born Augus..."
1,2,blake-poems.txt,Poems,0.0181,"...d we are put on earth a little space, That we may learn to bear the beams of love And these black bodies and this sunburnt face Is but a cloud, and like..."
2,3,austen-sense.txt,Sense and Sensibility,0.0172,"...xistence. By a former marriage, Mr. Henry Dashwood had one son: by his present lady, three daughters. The son, a steady respectable young man, was amply provi..."
3,4,shakespeare-macbeth.txt,Macbeth,0.0136,...Flourish. Exeunt. Scena Quinta. Enter Macbeths Wife alone with a Letter. Lady. They met me in the day of successe: and I haue learn'd by the perfect'st r...
4,5,austen-emma.txt,Emma,0.0129,"...him to Miss Churchill, of a great Yorkshire family, and Miss Churchill fell in love with him, nobody was surprized, except her brother and his wife, who had ne..."



q3: Tragedias sobre asesinato, ambición y realeza
    consulta: 'murder ambition king queen crown tragedy'


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,Hamlet,0.0250,...there? Fran. Nay answer me: Stand & vnfold your selfe Bar. Long liue the King Fran. Barnardo? Bar. He Fran. You come most carefully vpon your h...
1,2,carroll-alice.txt,Alice's Adventures in Wonderland,0.0172,"...o the other, saying, in a solemn tone, 'For the Duchess. An invitation from the Queen to play croquet.' The Frog-Footman repeated, in the same solemn tone, only..."
2,3,shakespeare-macbeth.txt,Macbeth,0.0131,"...ugh the fogge and filthie ayre. Exeunt. Scena Secunda. Alarum within. Enter King Malcome, Donalbaine, Lenox, with attendants, meeting a bleeding Captaine...."
3,4,bryant-stories.txt,Stories to Tell to Children,0.0109,"...eat big jungle; and in the jungle there was a great big Lion; and the Lion was king of the jungle. Whenever he wanted anything to eat, all he had to do was to..."
4,5,bible-kjv.txt,The King James Bible,0.0091,...[The King James Bible] The Old Testament of the King James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and th...



q4: Textos religiosos o épicos sobre la creación y la caída del hombre
    consulta: 'god creation heaven earth fall adam eve'


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,milton-paradise.txt,Paradise Lost,0.0825,"...sert Eternal Providence, And justify the ways of God to men. Say first--for Heaven hides nothing from thy view, Nor the deep tract of Hell--say first what..."
1,2,bible-kjv.txt,The King James Bible,0.0201,"...ng James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and v..."
2,3,blake-poems.txt,Poems,0.0169,"...on me, on all, Who became an infant small; Infant smiles are his own smiles; Heaven and earth to peace beguiles. DIVINE IMAGE To Mercy, Pity, Peace,..."
3,4,chesterton-ball.txt,The Ball and the Cross,0.0100,...s something stranger than hell itself; it is perhaps the last of the secrets of God. At the highest crisis of some incurable anguish there will suddenly fall u...
4,5,whitman-leaves.txt,Leaves of Grass,0.0097,"..., in other spheres, There to some group of mates the chants resuming, (Tallying Earth's soil, trees, winds, tumultuous waves,) Ever with pleas'd smile I may kee..."
